# DAI Mission - Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is Our team's mission proposal. 

> **Team size:** 3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Leila Rahimiyadkuri||
| Member | Forough Asgari| |
| Member | Sara Davoodabadi | |


## 2. Mission Title & Research Question

**Title:**  
Predicting and Explaining Customer Churn in Telecommunications: Combining Causal Inference, Supervised Classification, and Customer Segmentation

**Research question:**  
Does the type of contract a telecom customer holds (*month-to-month* vs. *longer-term*) causally increase their probability of churning and does this effect differ across distinct customer segments identified through unsupervised learning?

**Why it matters:**  
Telecom companies face high customer turnover, and retaining an existing customer is significantly cheaper than acquiring a new one. Most churn models tell you *who* is likely to leave, but not *why*. We want to separate correlation from causation: if month-to-month contracts genuinely *cause* higher churn (rather than simply being chosen by customers who were already planning to leave), then offering contract upgrades or lock-in incentives becomes a concrete, cost-effective retention lever. That distinction between prediction and causal mechanism is what drives our methodological design.

## 3. Data

**Source:**  
IBM Telco Customer Churn Dataset 
publicly available on Kaggle, originally published by IBM as a sample dataset for business analytics demonstrations.  
Link: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Unit of observation:** One row = one individual telecom customer. The dataset contains 7,043 rows and 21 columns.

**Key variables:**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| customerID | String | Identifier (excluded) | Unique customer ID - carries no predictive or causal information |
| Churn | Binary (Yes/No) | Target (Y) | Whether the customer left the company (Yes = churned) |
| Contract | Categorical | Treatment (W₁) | Contract type: Month-to-month, One year, Two year - our primary treatment variable |
| MonthlyCharges | Float | Treatment (W₂) | Amount charged to the customer per month |
| tenure | Integer | Confounder (X₁) | Number of months the customer has been with the company - affects both contract choice and churn likelihood |
| SeniorCitizen | Binary (0/1) | Confounder (X₂) | Whether the customer is a senior citizen - affects both spending patterns and loyalty |
| Partner | Binary (Yes/No) | Confounder (X₃) | Whether the customer has a partner - household situation affects contract preferences |
| Dependents | Binary (Yes/No) | Confounder (X₄) | Whether the customer has dependents - family situation affects switching costs |
| InternetService | Categorical | Feature | Type of internet service: DSL, Fiber optic, or None |
| OnlineSecurity | Categorical |  Mediator (excluded from DAG adjustment) | Add-on service - sits between Contract and Churn, conditioning on it would block the causal path |
| TechSupport | Categorical |  Mediator (excluded from DAG adjustment) | Same as above |
| StreamingTV | Categorical |  Mediator (excluded from DAG adjustment) | Same as above |
| StreamingMovies | Categorical |  Mediator (excluded from DAG adjustment) | Same as above |
| OnlineBackup | Categorical |  Mediator (excluded from DAG adjustment) | Same as above |
| DeviceProtection | Categorical |  Mediator (excluded from DAG adjustment) | Same as above |
| PhoneService | Binary (Yes/No) | Feature | Whether the customer has phone service |
| MultipleLines | Categorical | Feature | Whether the customer has multiple lines |
| PaperlessBilling | Binary (Yes/No) | Feature | Whether the customer uses paperless billing |
| PaymentMethod | Categorical | Feature | Payment method used by the customer |
| TotalCharges | Float | Feature | Total amount charged over the customer's lifetime - note: stored as string in raw data, requires conversion |

**Data quality & composition - known issues and mitigation plan:**

Even though this is a well-known Kaggle dataset, it has several concrete issues we will address explicitly:

1. **`TotalCharges` stored as string:** The column appears numeric but is of type `object` in the raw CSV, and contains 11 blank strings for customers with `tenure = 0`. We will convert using `pd.to_numeric(..., errors='coerce')` and impute or drop the 11 affected rows (they represent ~0.15% of data).

2. **`No internet service` as a categorical level:** Several add-on columns (e.g. `OnlineSecurity`, `TechSupport`) use `"No internet service"` as a third category instead of simply `"No"`. This is redundant and can confuse models. We will collapse these to binary Yes/No before encoding.

3. **Class imbalance (~26% churn rate):** The dataset is imbalanced - roughly 1 in 4 customers churned. We will not oversample, but will use `class_weight='balanced'` in our classifiers and report AUC-ROC and F1-Score rather than raw accuracy.

4. **Potential collection bias:** This dataset was originally created by IBM for demonstration purposes, not from a real anonymized telco. Features may be cleaner than real-world data, and the churn patterns may not generalize. We will discuss this limitation explicitly.

In [2]:
# ── Proposal code block ───────────────────────────────────────────────
import kagglehub
import pandas as pd
import numpy as np
import os

# Download dataset
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
df = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))

# --- Fix TotalCharges type issue
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# --- Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# --- Shape & types
print("--- Dataset Shape ---")
print(df.shape)

print("\n--- Column Types & Missing Values ---")
print(df.info())

print("\n--- Summary Statistics ---")
print(df.describe().T)

print("\n--- Churn Rate ---")
print(df['Churn'].value_counts(normalize=True).rename({0: 'Stayed', 1: 'Churned'}))

100%|██████████| 172k/172k [00:00<00:00, 1.44MB/s]

Extracting files...


--- Dataset Shape ---
(7043, 21)

--- Column Types & Missing Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract         

## 4. Planned Methods

### 4a. Causal Inference

-  Causal graph / DAG (DoWhy)
-  Backdoor adjustment


**Justification:**  
The central confounding problem here is that customers who choose month-to-month contracts are likely different from those who commit to longer contracts; they may be newer, less settled, or more price-sensitive to begin with. Simply comparing churn rates across contract types would mix up the effect of the contract itself with the pre-existing characteristics of the customer who chose it.

To isolate the genuine causal effect, we will construct a DAG using DoWhy with the following structure:

```
tenure          → Contract, Churn
SeniorCitizen   → Contract, Churn
Partner         → Contract, Churn
Dependents      → Contract, Churn
Contract        → Churn
MonthlyCharges  → Churn
```

We will apply **backdoor adjustment**, conditioning only on pre-contract confounders: `tenure`, `SeniorCitizen`, `Partner`, and `Dependents`. We deliberately exclude service add-on variables (`OnlineSecurity`, `TechSupport`, `StreamingTV`, `OnlineBackup`, `DeviceProtection`, `StreamingMovies`) from the adjustment set because they are **mediators** — they sit on the causal path between Contract and Churn. Conditioning on them would block part of the very effect we are trying to measure and bias our estimate.

We will validate the causal estimate using two DoWhy refutation tests:
1. **Random common cause** - injects a random confounder; the ATE should remain stable
2. **Placebo treatment** - replaces Contract with a random variable; the ATE should collapse to near zero

---

### 4b. Supervised Learning

-  Logistic regression
-  Decision Tree / Random Forest

**Justification:**  
Churn is a binary outcome, so we frame this as a classification problem. We start with **logistic regression** as an interpretable baseline - it gives us log-odds coefficients for each feature that we can compare directly against our causal estimates, which is a useful sanity check. We then train a **Random Forest** to capture non-linear interactions (for example, a month-to-month customer with fiber optic internet and high monthly charges may have a disproportionately high churn risk that a linear model would miss).

Both models will use stratified k-fold cross-validation and `class_weight='balanced'` to handle the imbalance. Feature importance from the Random Forest will also inform which variables drive predictive accuracy versus which ones matter causally - comparing these two lists is itself an interesting finding.

---

### 4c. Unsupervised Learning

-  K-Means clustering


**Justification:**  
Telecom customers are not a uniform group. A senior citizen on a month-to-month plan with fiber optic internet is likely a very different churn profile than a young customer on a two-year plan with basic DSL. Before fitting global models, we want to see if the data naturally separates into meaningful customer types.

We will apply **K-Means clustering** on a scaled subset of features (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`, encoded `Contract`) and select k using the elbow method and silhouette score. The resulting cluster labels serve two purposes: added as a feature in the supervised classifiers, and used as a stratification variable in the causal block to test whether the effect of contract type on churn is homogeneous across all segments or significantly stronger in some groups than others.

## 5. Evaluation Strategy

**Predictive models:**  
We will report AUC-ROC as our primary metric since it is robust to class imbalance. We will also report F1-Score and Recall — Recall is particularly important here because failing to identify a churner (false negative) is more costly for the company than a false alarm. All models are benchmarked against a dummy classifier that always predicts the majority class.

**Causal estimates:**  
We will run two DoWhy refutation tests:
1. *Random common cause* - adds a random variable as a fake confounder; our ATE estimate should not shift significantly
2. *Placebo treatment* - replaces Contract with a random variable; the estimated effect should collapse to near zero

Passing both tests gives us reasonable confidence that the DAG is correctly specified and the backdoor criterion is satisfied.

**Clustering:**  
We will select k using the elbow plot and silhouette coefficient. Each cluster will be interpreted descriptively — what kind of customer does each group represent, and does that group show a significantly different churn rate?

**Cross-block synthesis:**  
The central question connecting all three blocks is: does the causal effect of contract type on churn differ by customer segment? If K-Means reveals a segment where the ATE is particularly high, that identifies a concrete target group for retention interventions.

## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | All Members | Data collection & cleaning |
| 2 | Leila | EDA |
| 3 | Leila | Causal inference block |
| 4 | Sara | Supervised learning block |
| 5 | Forough | Unsupervised / generative block |
| 6 | All Members| Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
